# Thêm Thư Viện

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [ ]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

: 

# Đọc data

## Đọc data từ SQL Server

In [ ]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY So_the
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [ ]:
query_Bandoc = """
SELECT dbo.DecodeUTF8String(So_the) AS So_the,
       dbo.DecodeUTF8String(Ho_ten) AS Ho_ten,
       Ngay_sinh,
       Dan_toc_ID,
       Trinh_do_ID,
       dbo.DecodeUTF8String(So_dien_thoai) AS So_dien_thoai,
       dbo.DecodeUTF8String(Nghe_nghiep) AS Nghe_nghiep,
       dbo.DecodeUTF8String(Co_quan) AS Co_quan,
       dbo.DecodeUTF8String(chuc_vu) AS chuc_vu,
       dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
       dbo.DecodeUTF8String(Dia_chi_thuong_tru) AS Dia_chi_thuong_tru,
       dbo.DecodeUTF8String(Khoa_hoc) AS Khoa_hoc,
       Lop,
       Anh,
       Ngay_cap,
       Ngay_het_han,
       Email,
       Nhom_ID,
       Nhom_nghanh_nghe_ID,
       Gioi_tinh,
       Status,
       dbo.DecodeUTF8String(Ghi_chu) AS Ghi_chu,
       Mat_khau
FROM Ban_doc
"""
df_bandoc = fetch_data_in_batches(query_Bandoc, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_bandoc) # Hiển thị kết quả

C:\Users\phung\AppData\Local\Temp\ipykernel_17504\705386798.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


Lỗi xảy ra khi xử lý batch từ 61300: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 61600: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 61700: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 61800: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 62200: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 62400: ('42000', '[42000] [Microsof

C:\Users\phung\AppData\Local\Temp\ipykernel_17504\705386798.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


# Xử lý data

## Thêm 1 dòng giả định none

In [ ]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'So_the': ['0'],
    'Ho_ten': ['(Không xác định)'],
    'Ngay_sinh': ['1024-01-01 00:00:00'],
    'ID_nien_khoa': [0],
    'ID_dan_toc': [56],
    'ID_trinh_do': [0],
    'ID_lop': [0],
    'Ngay_cap': ['1024-01-01 00:00:00'],
    'Ngay_het_han': ['1024-01-01 00:00:00'],
    'ID_nhom_ban_doc': [0],
    'ID_nhom_nghanh_nghe': [0],
})

# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_bandoc = pd.concat([df_bandoc, new_row], ignore_index=True) # Thêm vào dataframe
df_bandoc['So_the'] = df_bandoc['So_the'].astype(str)
df_bandoc = df_bandoc.sort_values(by="So_the", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_bandoc)

## Xử lý data rỗng hoặc " "

In [ ]:
df_bandoc = df_bandoc.replace(np.nan, None)
df_bandoc = df_bandoc.replace('', None)
print(df_bandoc[['Nghe_nghiep', 'Co_quan', 'chuc_vu']])

## Xử lý kiểu date

In [ ]:
query_date = "SELECT Date_key FROM DIM_date"
df_date = pd.read_sql(query_date, conn_dwh_library)

date_ids = set(df_date['Date_key'])

# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_bandoc['Ngay_sinh'] = pd.to_datetime(df_bandoc['Ngay_sinh'], errors='coerce')
df_bandoc['Ngay_cap'] = pd.to_datetime(df_bandoc['Ngay_cap'], errors='coerce')
df_bandoc['Ngay_het_han'] = pd.to_datetime(df_bandoc['Ngay_het_han'], errors='coerce')

df_bandoc['Ngay_sinh'] = df_bandoc['Ngay_sinh'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_cap'] = df_bandoc['Ngay_cap'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_het_han'] = df_bandoc['Ngay_het_han'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)

print(df_bandoc[['Ngay_sinh', 'Ngay_cap', 'Ngay_het_han']])

## Xử lý Nien_khoa

## Xử lý Dan_toc

In [ ]:
# đọc dữ liệu lấy từ bộ về
df_Data_Dim_Dan_toc = pd.read_csv("./data_dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
# đọc dữ liệu đã lưu trong sql server
query_Dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)

#tìm và mapping 2 bảng lại
df_mapping = df_Dan_toc.copy()
df_mapping['Mapping Mã'] = None
df_mapping['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_mapping['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_mapping['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_mapping['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_mapping.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_mapping.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_mapping.at[i, 'Mapping Mã'] = 56
print(df_mapping)

In [ ]:
map_dict = dict(zip(df_mapping['Id'], df_mapping['Mapping Mã'])) # Tạo map_dict để ánh xạ từ Id sang Mapping_Ma trong df_mapping

for index, row in df_bandoc.iterrows(): # Lặp qua từng dòng trong df_ban_doc để cập nhật Dan_toc_ID
    
    if not pd.isna(row['Dan_toc_ID']): # Kiểm tra nếu Dan_toc_ID rỗng (None hoặc NaN)
        if row['Dan_toc_ID'] in map_dict: # Kiểm tra nếu Dan_toc_ID có trong map_dict
            df_bandoc.at[index, 'Dan_toc_ID'] = map_dict[row['Dan_toc_ID']]# Nếu tìm thấy, thay thế bằng giá trị Mapping_Ma
        else:
            df_bandoc.at[index, 'Dan_toc_ID'] = 56 # Nếu không tìm thấy, gán Dan_toc_ID bằng 0
    else:
        df_bandoc.at[index, 'Dan_toc_ID'] = 56
print(df_bandoc['Dan_toc_ID'])

## Xử lý Trinh_do

In [ ]:
query_Trinhdo = "SELECT ID_trinh_do FROM DIM_Trinh_do"
df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_library)

trinhdo_ids = set(df_trinhdo['ID_trinh_do'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong trinhdo_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Trinh_do_ID'] = df_bandoc['Trinh_do_ID'].apply(lambda x: x if x in trinhdo_ids else 0)

print(df_bandoc['Trinh_do_ID'])

## Xử lý Lop

In [ ]:
query_lop = "SELECT ID_lop FROM DIM_Lop"
df_lop = pd.read_sql(query_lop, conn_dwh_library)

lop_ids = set(df_lop['ID_lop'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong lop_ids hay không. 
# Nếu có, chuyển sang dạng chữ in hoa (upper()), nếu không, gán giá trị bằng 0.
df_bandoc['Lop'] = df_bandoc['Lop'].str.upper()
df_bandoc['Lop'] = df_bandoc['Lop'].apply(lambda x: x.upper() if x in lop_ids else 0)

print(df_bandoc['Lop'])

## Xử lý Nhom_ban_doc

In [ ]:
query_Nhombandoc = "SELECT ID_nhom_ban_doc FROM DIM_Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_library)

nhombandoc_ids = set(df_nhombandoc['ID_nhom_ban_doc'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhombandoc_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_ID'] = df_bandoc['Nhom_ID'].apply(lambda x: x if x in nhombandoc_ids else 0)

print(df_bandoc['Nhom_ID'])

## Xử lý Nhom_nghanh_nghe

In [ ]:
query_Nhomnghanhnghe = "SELECT ID_nhom_nghanh_nghe FROM DIM_Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_lib)

nhomnghanhnghe_ids = set(df_nhomnghanhnge['ID_nhom_nghanh_nghe'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhomnghanhnghe_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_nghanh_nghe_ID'] = df_bandoc['Nhom_nghanh_nghe_ID'].apply(lambda x: x if x in nhomnghanhnghe_ids else 0)

print(df_bandoc['Nhom_nghanh_nghe_ID'])

## Xử lý duplicate cho primary key

In [ ]:
df_bandoc = df_bandoc.drop_duplicates(subset='So_the').reset_index(drop=True) 
print(df_bandoc)

## Load data

### [Nếu cần] Clear bảng

In [ ]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM DIM_Ban_doc"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [ ]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO DIM_Ban_doc (
                    ID_ban_doc, Ho_ten, Ngay_sinh, ID_nien_khoa, 
                    ID_dan_toc, ID_trinh_do,
                    So_dien_thoai, Nghe_nghiep, Co_quan, Chuc_vu,
                    Dia_chi_tam_tru, Dia_chi_thuong_tru, ID_lop,
                    Anh, Ngay_cap, Ngay_het_han, Email, ID_nhom_ban_doc,
                    ID_nhom_nghanh_nghe, Gioi_tinh, Tinh_trang, Ghi_chu, Mat_khau
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """

# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['So_the'], row['Ho_ten'], row['Ngay_sinh'], row['Dan_toc_ID'], row['Trinh_do_ID'],
        row['So_dien_thoai'], row['Nghe_nghiep'], row['Co_quan'], row['chuc_vu'],
        row['Dia_chi'], row['Dia_chi_thuong_tru'], row['Khoa_hoc'], row['Lop'],
        row['Anh'], row['Ngay_cap'], row['Ngay_het_han'], row['Email'], row['Nhom_ID'],
        row['Nhom_nghanh_nghe_ID'], row['Gioi_tinh'], row['Status'], row['Ghi_chu'], row['Mat_khau']
    )
    for index, row in df_bandoc.iterrows()
]

cursor_dwh.executemany(insert_query, data_to_insert) # Sử dụng executemany để chèn dữ liệu cùng lúc
conn_dwh_library.commit() # Commit thay đổi
cursor_dwh.close() # Đóng cursor và kết nối
conn_dwh_library.close()